# Data Collection and Exploration

## 1. Dataset Overview

This project investigates **musical-theatre lyricist identification** using song lyrics. The planned comparison includes word-level TF-IDF with logistic regression and pretrained (fixed) Transformer embeddings with logistic regression. Evaluation will consider both song-level splits and held-out-musical splits to assess generalization beyond individual shows.

The four target lyricists are **Stephen Sondheim, Stephen Schwartz, Jason Robert Brown, and Oscar Hammerstein II**. 

The collection target is **4–6 musicals per lyricist, with six preferred**, depending on lyric availability, verified authorship, and the time required for manual cleaning.

The goal is to collect enough independent musicals to support meaningful held-out-musical evaluation. This approach will help determine whether models learn generalizable characteristics of a lyricist's writing rather than vocabulary or themes associated with individual shows.

Song inclusion depends on verified lyricist credits and the availability of suitable sung material. Spoken dialogue, stage directions, and material not originally written by the attributed lyricist will be excluded.

### Selected Candidate Musicals

The working collection target is **six musicals per lyricist**, for a total of **24 musicals**. These selections are provisional and depend on verified lyric credits, available song counts, and suitable sung material.

| Lyricist | Musical | Status |
|---|---|---|
| **Stephen Sondheim** | Company | Candidate |
| | Follies | Candidate |
| | Sweeney Todd | Candidate |
| | Sunday in the Park with George | Candidate |
| | Into the Woods | Candidate |
| | Assassins | Candidate |
| **Stephen Schwartz** | Godspell | Candidate |
| | Pippin | Candidate |
| | The Baker's Wife | Candidate |
| | Children of Eden | Candidate |
| | Wicked | Candidate |
| | Schikaneder | Candidate |
| **Jason Robert Brown** | Songs for a New World | **Completed:** 16 songs |
| | Parade | **Completed:** 28 songs |
| | The Last Five Years | **Completed:** 14 songs |
| | 13 | Candidate |
| | The Bridges of Madison County | Candidate |
| | Honeymoon in Vegas | Candidate |
| **Oscar Hammerstein II** | Show Boat | Candidate |
| | Oklahoma! | Candidate |
| | Carousel | Candidate |
| | South Pacific | Candidate |
| | The King and I | Candidate |
| | The Sound of Music | Candidate |

#### Collection Progress

- **Target:** 24 musicals (6 per lyricist)
- **Completed:** 3 musicals
- **Remaining:** 21 musicals
- **Included songs collected:** 58

#### Proposed Evaluation Strategy

With six musicals per lyricist, the dataset will support a **show-level 4/2 train/test split**:

- **Training:** 4 musicals per lyricist (16 total)
- **Testing:** 2 held-out musicals per lyricist (8 total)

This approach evaluates whether models learn generalizable characteristics of each lyricist's writing rather than show-specific vocabulary or themes.

Additional candidate musicals will remain in `musical_inventory.csv` as backup options if the selected musicals have insufficient suitable material.

## 2. Dataset Organization

Two levels of metadata are maintained:

- `data/musical_inventory.csv`: musical-level information, including lyricist, production details, credits, song counts, and lyric sources.
- `data/raw/jason_robert_brown/jrb_inventory.csv`: JRB song-level metadata, including `song_id`, `lyricist`, `musical`, `production_year`, `song_title`, `source_url`, `lyrics_source`, `lyrics_collected`, and `include_in_dataset`.

Song lyrics are stored separately as UTF-8 text files rather than duplicated in the inventory. Raw files are retained under `data/raw/`, and manually verified lyrics are stored in `data/processed/`, with subfolders for each lyrisist and musical. Filenames match the `song_id` (for example, `JRB-PAR-011.txt`).

### Lyric Collection, Cleaning, and Verification

The first completed collection focuses on Jason Robert Brown's *Songs for a New World*, *The Last Five Years*, and *Parade*. Source URLs and lyric-credit information are recorded in the song inventory. The original text is kept separately from the verified processed text.

The inventory initially listed **61 tracks**. Following review, **58 songs** are intended for the authorship dataset, with **three exclusions**. These counts are expected from the current work and are verified below against the live inventory.

Initial automated cleaning removed bracketed annotations, some speaker labels, and extra whitespace. Crucially, automated rules **cannot reliably distinguish spoken dialogue from sung text**. Every JRB track was therefore manually reviewed against a recording; spoken dialogue was removed while sung words and meaningful repetitions were preserved. Original capitalization and wording remain in the processed files, with normalization deferred to model-specific preprocessing.

## 2. Exploratory Data Analysis

Word counts include repeated words. Vocabulary size counts distinct lowercase tokens; these operations are for **analysis only** and do not change the stored lyric files. The simple regex tokenizer is not intended to match every model's tokenizer.

#### Jason Robert Brown EDA

In [ ]:
import re
import pandas as pd
from IPython.display import display
from pathlib import Path

# Paths
PROJECT_ROOT = Path("..")
INVENTORY_PATH = PROJECT_ROOT / "data/raw/jason_robert_brown/jrb_inventory.csv"
PROCESSED_DIR = PROJECT_ROOT / "data/processed/jason_robert_brown"

# Load song inventory
inventory = pd.read_csv(INVENTORY_PATH)

# Keep only songs marked for inclusion
included = inventory[
    inventory["include_in_dataset"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
].copy()

# Map each song ID to its processed lyrics file
file_map = {
    path.stem: path
    for path in PROCESSED_DIR.rglob("*.txt")
}

print("Included songs:", len(included))
print("Processed lyric files:", len(file_map))

missing = set(included["song_id"]) - set(file_map)

print("Missing lyric files:", len(missing))

if missing:
    print(sorted(missing))

Included songs: 58
Processed lyric files: 58
Missing lyric files: 0


In [ ]:
records = []
for _, row in included.iterrows():
    song_id = str(row['song_id'])
    if song_id not in file_map:
        continue
    text = file_map[song_id].read_text(encoding='utf-8')
    tokens = re.findall(r"\b[\w']+\b", text.lower())
    records.append({
        'song_id': song_id,
        'lyricist': row['lyricist'],
        'musical': row['musical'],
        'word_count': len(tokens),
        'unique_words': len(set(tokens)),
        'text': text,
    })
stats_df = pd.DataFrame(records)
if stats_df.empty:
    raise ValueError('No included processed songs found.')

print('Songs analyzed:', len(stats_df))
print('Total words:', f"{stats_df['word_count'].sum():,}")
print('Average words per song:', f"{stats_df['word_count'].mean():.1f}")
print('Shortest song:', stats_df['word_count'].min())
print('Longest song:', stats_df['word_count'].max())
display(stats_df.drop(columns='text').head())

Songs analyzed: 58
Total words: 19,011
Average words per song: 327.8
Shortest song: 24
Longest song: 737


,song_id,lyricist,musical,word_count,unique_words
0,JRB-SNW-001,Jason Robert Brown,Songs for a New World,391,152
1,JRB-SNW-002,Jason Robert Brown,Songs for a New World,462,137
2,JRB-SNW-003,Jason Robert Brown,Songs for a New World,422,204
3,JRB-SNW-004,Jason Robert Brown,Songs for a New World,343,141
4,JRB-SNW-005,Jason Robert Brown,Songs for a New World,534,163


In [ ]:
musical_stats = (
    stats_df.groupby(['lyricist', 'musical'])
    .agg(song_count=('song_id', 'count'),
         total_words=('word_count', 'sum'),
         avg_words_per_song=('word_count', 'mean'),
         min_words=('word_count', 'min'),
         max_words=('word_count', 'max'))
    .reset_index()
)
vocab = []
for (lyricist, musical), group in stats_df.groupby(['lyricist', 'musical']):
    tokens = re.findall(r"\b[\w']+\b", ' '.join(group['text']).lower())
    vocab.append({'lyricist': lyricist, 'musical': musical, 'vocabulary_size': len(set(tokens))})
musical_stats = musical_stats.merge(pd.DataFrame(vocab), on=['lyricist', 'musical'])
display(musical_stats.round(1))

,lyricist,musical,song_count,total_words,avg_words_per_song,min_words,max_words,vocabulary_size
0,Jason Robert Brown,Parade,28,6737,240.6,24,706,1296
1,Jason Robert Brown,Songs for a New World,16,6514,407.1,249,573,1182
2,Jason Robert Brown,The Last Five Years,14,5760,411.4,193,737,1251


In [ ]:
short_songs = stats_df.loc[stats_df['word_count'] < 100,
                                ['song_id', 'musical', 'word_count', 'unique_words']].sort_values('word_count')
print('Songs with fewer than 100 words:', len(short_songs))
display(short_songs)

Songs with fewer than 100 words: 4


,song_id,musical,word_count,unique_words
38,JRB-PAR-009,Parade,24,18
48,JRB-PAR-020,Parade,36,25
43,JRB-PAR-015,Parade,37,26
31,JRB-PAR-002,Parade,56,36
